# BLOSUM62 Encoding for PTM Prediction

This notebook encodes protein sequences using the BLOSUM62 substitution matrix.

## What is BLOSUM62?

**BLOSUM62** (BLOcks SUbstitution Matrix) is a substitution matrix used to score alignments between evolutionarily divergent protein sequences.

- Each amino acid is represented by a **20-dimensional vector**
- Values represent **log-odds scores** for amino acid substitutions
- Positive scores = likely substitution, Negative scores = unlikely substitution

## Why BLOSUM for PTM Prediction?

✓ **Captures evolutionary relationships** between amino acids
✓ **Proven to work best** (from teammate's research)
✓ **Better than one-hot** which treats all AAs as equally different
✓ Example: BLOSUM knows C and S are similar (both polar), but C and W are different

## Encoding Method

For a 31-residue sequence:
- Each position → 20 BLOSUM scores
- Total features: 31 × 20 = **620 features**
- Plus original physicochemical features = ~**655 total features**

## Configuration

In [9]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# File paths
INPUT_FILE = "../data_engineered/train_with_features.csv"  # From notebook 02
OUTPUT_FILE = "../data_engineered/blosum/train_with_features_blosum.csv"

# Sequence parameters
MAX_LENGTH = 31  # All sequences are 31 amino acids

# Processing
CHUNK_SIZE = 10000  # Process in chunks to save memory

print("Configuration loaded:")
print(f"  Input: {INPUT_FILE}")
print(f"  Output: {OUTPUT_FILE}")
print(f"  Max length: {MAX_LENGTH}")
print(f"  Chunk size: {CHUNK_SIZE}")

Configuration loaded:
  Input: ../data_engineered/train_with_features.csv
  Output: ../data_engineered/blosum/train_with_features_blosum.csv
  Max length: 31
  Chunk size: 10000


## Import Libraries

In [3]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

print("✓ Libraries imported")

✓ Libraries imported


## BLOSUM62 Matrix

In [4]:
# BLOSUM62 substitution matrix
# Rows/Columns: A  R  N  D  C  Q  E  G  H  I  L  K  M  F  P  S  T  W  Y  V
BLOSUM62 = {
    'A': [ 4, -1, -2, -2,  0, -1, -1,  0, -2, -1, -1, -1, -1, -2, -1,  1,  0, -3, -2,  0],
    'R': [-1,  5,  0, -2, -3,  1,  0, -2,  0, -3, -2,  2, -1, -3, -2, -1, -1, -3, -2, -3],
    'N': [-2,  0,  6,  1, -3,  0,  0,  0,  1, -3, -3,  0, -2, -3, -2,  1,  0, -4, -2, -3],
    'D': [-2, -2,  1,  6, -3,  0,  2, -1, -1, -3, -4, -1, -3, -3, -1,  0, -1, -4, -3, -3],
    'C': [ 0, -3, -3, -3,  9, -3, -4, -3, -3, -1, -1, -3, -1, -2, -3, -1, -1, -2, -2, -1],
    'Q': [-1,  1,  0,  0, -3,  5,  2, -2,  0, -3, -2,  1,  0, -3, -1,  0, -1, -2, -1, -2],
    'E': [-1,  0,  0,  2, -4,  2,  5, -2,  0, -3, -3,  1, -2, -3, -1,  0, -1, -3, -2, -2],
    'G': [ 0, -2,  0, -1, -3, -2, -2,  6, -2, -4, -4, -2, -3, -3, -2,  0, -2, -2, -3, -3],
    'H': [-2,  0,  1, -1, -3,  0,  0, -2,  8, -3, -3, -1, -2, -1, -2, -1, -2, -2,  2, -3],
    'I': [-1, -3, -3, -3, -1, -3, -3, -4, -3,  4,  2, -3,  1,  0, -3, -2, -1, -3, -1,  3],
    'L': [-1, -2, -3, -4, -1, -2, -3, -4, -3,  2,  4, -2,  2,  0, -3, -2, -1, -2, -1,  1],
    'K': [-1,  2,  0, -1, -3,  1,  1, -2, -1, -3, -2,  5, -1, -3, -1,  0, -1, -3, -2, -2],
    'M': [-1, -1, -2, -3, -1,  0, -2, -3, -2,  1,  2, -1,  5,  0, -2, -1, -1, -1, -1,  1],
    'F': [-2, -3, -3, -3, -2, -3, -3, -3, -1,  0,  0, -3,  0,  6, -4, -2, -2,  1,  3, -1],
    'P': [-1, -2, -2, -1, -3, -1, -1, -2, -2, -3, -3, -1, -2, -4,  7, -1, -1, -4, -3, -2],
    'S': [ 1, -1,  1,  0, -1,  0,  0,  0, -1, -2, -2,  0, -1, -2, -1,  4,  1, -3, -2, -2],
    'T': [ 0, -1,  0, -1, -1, -1, -1, -2, -2, -1, -1, -1, -1, -2, -1,  1,  5, -2, -2,  0],
    'W': [-3, -3, -4, -4, -2, -2, -3, -2, -2, -3, -2, -3, -1,  1, -4, -3, -2, 11,  2, -3],
    'Y': [-2, -2, -2, -3, -2, -1, -2, -3,  2, -1, -1, -2, -1,  3, -3, -2, -2,  2,  7, -1],
    'V': [ 0, -3, -3, -3, -1, -2, -2, -3, -3,  3,  1, -2,  1, -1, -2, -2,  0, -3, -1,  4]
}

AA_ORDER = list("ARNDCQEGHILKMFPSTWYV")  # Order of AAs in matrix
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_ORDER)}

print("✓ BLOSUM62 matrix loaded")
print(f"  Matrix size: 20 × 20")
print(f"  Amino acids: {AA_ORDER}")

✓ BLOSUM62 matrix loaded
  Matrix size: 20 × 20
  Amino acids: ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']


## Encoding Functions

In [5]:
def sequence_to_blosum(sequence, max_length=31):
    """
    Encode a single sequence using BLOSUM62.
    
    Args:
        sequence: Protein sequence string
        max_length: Maximum sequence length (pad/truncate)
        
    Returns:
        1D array of BLOSUM features (max_length * 20)
    """
    # Initialize matrix: (max_length, 20)
    blosum_matrix = np.zeros((max_length, 20), dtype=np.float32)
    
    # Encode each position
    for i, aa in enumerate(sequence[:max_length]):
        if aa in BLOSUM62:
            blosum_matrix[i, :] = BLOSUM62[aa]
        # If unknown AA, leave as zeros
    
    # Flatten to 1D vector
    return blosum_matrix.flatten()  # Shape: (620,)

def encode_chunk(chunk_df, max_length=31):
    """
    Encode a chunk of sequences using BLOSUM62.
    
    Returns DataFrame with original features + BLOSUM features.
    """
    sequences = chunk_df['Sequence'].values
    
    # Encode all sequences in chunk
    blosum_features = np.vstack([
        sequence_to_blosum(seq, max_length)
        for seq in sequences
    ])
    
    # Create column names: blosum_pos0_A, blosum_pos0_R, ..., blosum_pos30_V
    blosum_cols = []
    for pos in range(max_length):
        for aa in AA_ORDER:
            blosum_cols.append(f"blosum_pos{pos}_{aa}")
    
    # Create DataFrame with BLOSUM features
    blosum_df = pd.DataFrame(blosum_features, columns=blosum_cols, index=chunk_df.index)
    
    # Combine with original features (keep everything except Sequence)
    # Drop Sequence column to save space
    result_df = pd.concat([chunk_df.drop(columns=['Sequence']), blosum_df], axis=1)
    
    return result_df

print("✓ Encoding functions defined")

✓ Encoding functions defined


## Load and Encode Data

In [6]:
print("="*80)
print("LOADING AND ENCODING DATA")
print("="*80)

# Check if input file exists
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}\n"
        "Please run 02_feature_engineering.ipynb first."
    )

# Get total rows
total_rows = sum(1 for _ in open(INPUT_FILE)) - 1  # -1 for header
print(f"\nTotal sequences to encode: {total_rows:,}")

# Process in chunks
chunks_processed = []
for chunk in tqdm(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE), 
                   total=(total_rows // CHUNK_SIZE) + 1,
                   desc="Encoding chunks"):
    
    encoded_chunk = encode_chunk(chunk, max_length=MAX_LENGTH)
    chunks_processed.append(encoded_chunk)

# Combine all chunks
print("\nCombining chunks...")
encoded_df = pd.concat(chunks_processed, ignore_index=True)

print(f"\n✓ Encoding complete!")
print(f"  Total samples: {len(encoded_df):,}")
print(f"  Total features: {len(encoded_df.columns)}")
print(f"  BLOSUM features: 620")
print(f"  Original features: {len(encoded_df.columns) - 620}")

LOADING AND ENCODING DATA

Total sequences to encode: 89,010


Encoding chunks: 100%|█████████████████████████████████████████████████████████| 9/9 [00:04<00:00,  1.83it/s]



Combining chunks...

✓ Encoding complete!
  Total samples: 89,010
  Total features: 658
  BLOSUM features: 620
  Original features: 38


## Verify Encoding

In [7]:
print("\n" + "="*80)
print("VERIFICATION")
print("="*80)

# Check for NaN/Inf values
nan_count = encoded_df.isnull().sum().sum()
inf_count = np.isinf(encoded_df.select_dtypes(include=[np.number])).sum().sum()

print(f"\nData quality:")
print(f"  NaN values: {nan_count}")
print(f"  Inf values: {inf_count}")

if nan_count > 0 or inf_count > 0:
    print("  ⚠️  Warning: Found invalid values!")
else:
    print("  ✓ All values are valid")

# Show sample
print(f"\nSample of encoded features:")
print(encoded_df[encoded_df.columns[:10]].head())

# Check BLOSUM features
blosum_cols = [col for col in encoded_df.columns if col.startswith('blosum_')]
print(f"\nBLOSUM features:")
print(f"  Count: {len(blosum_cols)}")
print(f"  Min value: {encoded_df[blosum_cols].min().min():.2f}")
print(f"  Max value: {encoded_df[blosum_cols].max().max():.2f}")
print(f"  Mean value: {encoded_df[blosum_cols].mean().mean():.2f}")


VERIFICATION

Data quality:
  NaN values: 0
  Inf values: 0
  ✓ All values are valid

Sample of encoded features:
       ID  S-glutathionylation  S-nitrosylation  S-palmitoylation      aa_A  \
0  Q9H4H8                    0                0                 0  0.193548   
1  Q8N697                    0                1                 0  0.419355   
2  Q8IP90                    0                0                 0  0.193548   
3  P36897                    0                0                 0  0.258065   
4  Q8NCN4                    0                0                 0  0.354839   

       aa_C      aa_D      aa_E      aa_F      aa_G  
0  0.032258  0.096774  0.096774  0.064516  0.064516  
1  0.032258  0.000000  0.064516  0.064516  0.096774  
2  0.032258  0.064516  0.096774  0.064516  0.096774  
3  0.129032  0.064516  0.000000  0.064516  0.032258  
4  0.032258  0.032258  0.032258  0.000000  0.129032  

BLOSUM features:
  Count: 620
  Min value: -4.00
  Max value: 11.00
  Mean value: -1.

## Save Encoded Data

In [10]:
print("\n" + "="*80)
print("SAVING ENCODED DATA")
print("="*80)

# Create output directory
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# Save to CSV
encoded_df.to_csv(OUTPUT_FILE, index=False)

file_size = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)  # MB

print(f"\n✓ Saved to: {OUTPUT_FILE}")
print(f"  File size: {file_size:.1f} MB")
print(f"  Samples: {len(encoded_df):,}")
print(f"  Features: {len(encoded_df.columns)}")

print("\n" + "="*80)
print("✓ BLOSUM ENCODING COMPLETE!")
print("="*80)
print(f"\nNext step: Run 04_train_val_split.ipynb with this file to create train/val splits.")
print(f"Or proceed directly to training with BLOSUM-encoded features!")


SAVING ENCODED DATA

✓ Saved to: ../data_engineered/blosum/train_with_features_blosum.csv
  File size: 281.3 MB
  Samples: 89,010
  Features: 658

✓ BLOSUM ENCODING COMPLETE!

Next step: Run 04_train_val_split.ipynb with this file to create train/val splits.
Or proceed directly to training with BLOSUM-encoded features!


## Example: How BLOSUM Works

```python
# Example sequence: "ACG"
# 
# Position 0: A → BLOSUM['A'] = [4, -1, -2, -2, 0, ...] (20 values)
# Position 1: C → BLOSUM['C'] = [0, -3, -3, -3, 9, ...] (20 values)
# Position 2: G → BLOSUM['G'] = [0, -2, 0, -1, -3, ...] (20 values)
# 
# Final encoding: 60 features (3 positions × 20 values)
```

### Why This is Better Than One-Hot:

**One-Hot**: C = [0,1,0,0,0,...] (just says "this is C")

**BLOSUM**: C = [0,-3,-3,-3,9,-3,-4,...] (says "C is similar to itself (9), dissimilar to R (-3), etc.")

The BLOSUM encoding captures **evolutionary relationships** which helps the model learn better patterns!